# 03: Pipeline RAG Completo y Validado

Este notebook demuestra el proceso completo de *Retrieval-Augmented Generation* (RAG) paso a paso, utilizando los servicios del Cognito-Stack.

**Pasos:**
1. Cargar documentos de ejemplo.
2. Generar embeddings usando un modelo de Ollama (`nomic-embed-text`).
3. Crear una colección en Qdrant e insertar los documentos vectorizados.
4. Realizar una consulta semántica para recuperar el contexto relevante.
5. Usar un LLM de Ollama (`llama3.2`) para generar una respuesta basada en el contexto recuperado.
6. Evaluar la calidad de la respuesta con métricas simples.

In [ ]:
# Celda 1: Configuración y Documentos de Ejemplo

import httpx
import asyncio
import os
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

# Cargar variables de entorno (asumiendo que se ejecuta desde la raíz del proyecto)
load_dotenv('mcp-server/.env')

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_NAME = "tarragona_projects"

print(f"Usando Ollama en: {OLLAMA_URL}")
print(f"Usando Qdrant en: {QDRANT_URL}")

# Documentos de ejemplo sobre proyectos de Tarragona
docs = [
    "El proyecto Tarragona Smart City implementa sensores IoT para monitorizar el tráfico y la calidad del aire.",
    "La iniciativa OpenData del ayuntamiento publica datasets municipales sobre transporte, censo y presupuesto.",
    "El sistema de gestión de residuos de la ciudad utiliza inteligencia artificial predictiva para optimizar las rutas de recogida.",
    "Se ha lanzado una nueva app ciudadana para reportar incidencias en el mobiliario urbano y el alumbrado público."
]

print(f"\n{len(docs)} documentos cargados para el ejemplo.")

In [ ]:
# Celda 2: Generar Embeddings con Ollama

async def generate_embeddings(texts):
    """Genera embeddings para una lista de textos usando el servicio de Ollama."""
    embeddings = []
    async with httpx.AsyncClient(timeout=60.0) as client:
        for text in texts:
            try:
                response = await client.post(
                    f"{OLLAMA_URL}/api/embeddings",
                    json={"model": "nomic-embed-text", "prompt": text}
                )
                response.raise_for_status()
                embeddings.append(response.json()["embedding"])
            except httpx.RequestError as e:
                print(f"Error al contactar con Ollama: {e}")
                return None
            except Exception as e:
                print(f"Error inesperado durante la generación de embeddings: {e}")
                return None
    return embeddings

print("Generando embeddings para los documentos...")
# Usamos await en el nivel superior (compatible con Jupyter/IPython)
embeddings = await generate_embeddings(docs)

if embeddings:
    print(f"✓ {len(embeddings)} embeddings generados. Tamaño del vector: {len(embeddings[0])}")

In [ ]:
# Celda 3: Insertar Documentos en Qdrant

client = QdrantClient(url=QDRANT_URL)

if embeddings:
    try:
        # (Re)crear la colección para asegurar un estado limpio
        client.recreate_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=models.VectorParams(
                size=len(embeddings[0]),
                distance=models.Distance.COSINE
            )
        )
        print(f"Colección '{COLLECTION_NAME}' creada en Qdrant.")

        # Crear puntos para insertar en Qdrant
        points = [
            models.PointStruct(
                id=idx,
                vector=emb,
                payload={"text": text}
            )
            for idx, (emb, text) in enumerate(zip(embeddings, docs))
        ]

        # Insertar los puntos en la colección
        client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
        print(f"✓ {len(points)} documentos insertados en Qdrant.")
        
    except Exception as e:
        print(f"Error al interactuar con Qdrant: {e}")

In [ ]:
# Celda 4: Realizar una Búsqueda Semántica

query = "¿Qué iniciativas hay sobre sensores inteligentes en la ciudad?"

print(f"Consulta del usuario: '{query}'")

# Generar el embedding para la consulta
query_embedding = await generate_embeddings([query])

if query_embedding:
    try:
        # Realizar la búsqueda en Qdrant
        search_results = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_embedding[0],
            limit=3,
            with_payload=True
        )

        print("\nDocumentos recuperados de Qdrant:")
        for result in search_results:
            print(f"  - Score: {result.score:.3f} | Documento: {result.payload['text']}")
            
    except Exception as e:
        print(f"Error durante la búsqueda en Qdrant: {e}")

In [ ]:
# Celda 5: Generar una Respuesta con el LLM

async def generate_answer(query, context):
    """Genera una respuesta usando el LLM con un contexto específico."""
    system_prompt = """Eres un asistente experto en Tarragona Connect OS.
Usa el contexto proporcionado para responder con precisión y de forma concisa.
Cita las fuentes o proyectos mencionados."""
    
    full_prompt = f"""Contexto de documentos recuperados:
{context}

Pregunta del usuario: {query}

Responde basándote únicamente en el contexto anterior."""
    
    async with httpx.AsyncClient(timeout=90.0) as client:
        try:
            response = await client.post(
                f"{OLLAMA_URL}/api/generate",
                json={
                    "model": "llama3.2",
                    "prompt": full_prompt,
                    "system": system_prompt,
                    "stream": False
                }
            )
            response.raise_for_status()
            return response.json()["response"]
        except httpx.RequestError as e:
            return f"Error al contactar con Ollama: {e}"
        except Exception as e:
            return f"Error inesperado al generar la respuesta: {e}"

# Construir el contexto a partir de los resultados de la búsqueda
if 'search_results' in locals() and search_results:
    context_for_llm = "\n".join([r.payload['text'] for r in search_results])
    
    print("Generando respuesta con el LLM...")
    answer = await generate_answer(query, context_for_llm)
    
    print(f"\n✓ Respuesta generada:\n---\n{answer}\n---")
else:
    print("No se encontraron resultados de búsqueda para generar una respuesta.")
    answer = ""

In [ ]:
# Celda 6: Métricas Simples de Calidad de la Respuesta

def evaluate_rag_response(query, context, answer):
    """Calcula métricas básicas de calidad para la respuesta del RAG."""
    if not answer or not context:
        return {"error": "Respuesta o contexto vacíos."}
    
    query_words = set(query.lower().split())
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())
    
    metrics = {
        "longitud_respuesta": len(answer.split()),
        "relevancia_respuesta_a_query": len(query_words.intersection(answer_words)) / len(query_words) if query_words else 0,
        "fundamentacion_en_contexto": len(answer_words.intersection(context_words)) / len(answer_words) if answer_words else 0,
        "palabras_clave_del_contexto_usadas": [word for word in ['iot', 'sensores', 'tráfico', 'residuos'] if word in answer.lower()]
    }
    return metrics

if answer:
    quality_metrics = evaluate_rag_response(query, context_for_llm, answer)
    print("\n📊 Métricas de Calidad de la Respuesta:")
    for key, value in quality_metrics.items():
        if isinstance(value, float):
            print(f"  - {key.replace('_', ' ').capitalize()}: {value:.2%}")
        else:
            print(f"  - {key.replace('_', ' ').capitalize()}: {value}")